In [ ]:
import pandas as pd
import re
from pathlib import Path

team_data_dir = Path("data/team-data")

# Already has all 16 target columns
akash_df = pd.read_json(team_data_dir / "akash.jsonl", lines=True)

# Format A: raw pod_describe / pod_logs / pod_logs_previous
mrunali_df = pd.read_json(team_data_dir / "mrunali.jsonl", lines=True)

# Format B: pre-parsed fields with evidence_text containing embedded describe
chels_df = pd.read_json(team_data_dir / "chels.jsonl", lines=True)
devesh_df = pd.read_json(team_data_dir / "devesh.jsonl", lines=True)
najel_df = pd.read_json(team_data_dir / "najel.jsonl", lines=True)

print(f"akash: {len(akash_df)}")
print(f"mrunali: {len(mrunali_df)}")
print(f"chels: {len(chels_df)}, devesh: {len(devesh_df)}, najel: {len(najel_df)}")
print(f"Total: {len(akash_df) + len(mrunali_df) + len(chels_df) + len(devesh_df) + len(najel_df)}")

In [ ]:
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.lower() in {"nan", "none"}:
        return ""
    x = x.replace("\r\n", "\n").replace("\r", "\n").replace("\t", "    ")
    lines = [line.rstrip() for line in x.split("\n")]
    return "\n".join(lines).strip()

def extract_first(pattern, text, flags=re.IGNORECASE | re.MULTILINE):
    if not text:
        return None
    m = re.search(pattern, text, flags)
    return m.group(1).strip() if m else None

def extract_block(text, block_name):
    if not text:
        return None
    lines = text.split("\n")
    block = []
    in_block = False
    for line in lines:
        if not in_block:
            if re.match(rf"^{re.escape(block_name)}:\s*$", line.strip()):
                in_block = True
            continue
        if line and not line.startswith(" ") and re.match(r"^[A-Za-z0-9_.\-/ ]+:\s*", line):
            break
        block.append(line)
    out = "\n".join(block).strip()
    return out if out else None

def parse_events_block(events_text):
    if not events_text:
        return {"event_reason": None, "event_message": None}
    reasons, messages = [], []
    lines = [line for line in events_text.split("\n") if line.strip()]
    for line in lines:
        if "Reason" in line and "Message" in line:
            continue
        if line.strip().startswith("----"):
            continue
        parts = re.split(r"\s{2,}", line.strip())
        if len(parts) >= 5:
            reasons.append(parts[1].strip())
            messages.append(parts[-1].strip())
    return {
        "event_reason": reasons[0] if reasons else None,
        "event_message": messages[0] if messages else None,
    }

def parse_describe(text):
    containers_block = extract_block(text, "Containers")
    volumes_block = extract_block(text, "Volumes")
    events_block = extract_block(text, "Events")
    events = parse_events_block(events_block)
    return {
        "pod_name": extract_first(r"^Name:\s*(.+)$", text),
        "namespace": extract_first(r"^Namespace:\s*(.+)$", text),
        "service_account_name": extract_first(r"^Service Account:\s*(.+)$", text),
        "node": extract_first(r"^Node:\s*(.+)$", text),
        "pod_status": extract_first(r"^Status:\s*(.+)$", text),
        "image": extract_first(r"^\s*Image:\s*(.+)$", containers_block or text),
        "container_state": extract_first(r"^\s*State:\s*(.+)$", containers_block or text),
        "last_state": extract_first(r"^\s*Last State:\s*(.+)$", containers_block or text),
        "ready": extract_first(r"^\s*Ready:\s*(.+)$", containers_block or text),
        "restart_count": extract_first(r"^\s*Restart Count:\s*(.+)$", containers_block or text),
        "node_selectors": extract_first(r"^Node-Selectors:\s*(.+)$", text),
        "claim_name": extract_first(r"^\s*ClaimName:\s*(.+)$", volumes_block or ""),
        "event_reason": events["event_reason"],
        "event_message": events["event_message"],
    }

def build_evidence_text(row):
    parts = []
    if row.get("pod_describe"):
        parts.append("POD DESCRIBE:\n" + row["pod_describe"])
    if row.get("pod_logs"):
        parts.append("POD LOGS:\n" + row["pod_logs"])
    if row.get("pod_logs_previous"):
        parts.append("POD LOGS PREVIOUS:\n" + row["pod_logs_previous"])
    return "\n\n".join(parts).strip() if parts else None

def extract_describe_from_evidence(evidence_text):
    if not evidence_text:
        return ""
    m = re.search(r"=== kubectl describe pod ===\n(.*?)(?:\n=== |\Z)", evidence_text, re.DOTALL)
    return m.group(1).strip() if m else ""

In [ ]:
# --- Process Format A (mrunali): parse pod_describe into structured fields ---
raw_cols = ["scenario_id", "pod_describe", "pod_logs", "pod_logs_previous"]
format_a = mrunali_df[raw_cols].copy()

for col in ["pod_describe", "pod_logs", "pod_logs_previous"]:
    format_a[col] = format_a[col].apply(normalize_text)

parsed_a = format_a["pod_describe"].apply(parse_describe).apply(pd.Series)
format_a = pd.concat([format_a, parsed_a], axis=1)
format_a["evidence_text"] = format_a.apply(build_evidence_text, axis=1)

print(f"Format A (mrunali): {len(format_a)} rows")
format_a[["scenario_id", "pod_name", "namespace", "pod_status", "event_reason"]].head()

In [ ]:
# --- Process Format B (chels, devesh, najel): extract describe from evidence_text, parse missing fields ---
format_b = pd.concat([chels_df, devesh_df, najel_df], ignore_index=True)
format_b["evidence_text"] = format_b["evidence_text"].apply(normalize_text)

# Extract pod_describe, pod_logs, pod_logs_previous from evidence_text
def extract_section(evidence_text, section_name):
    if not evidence_text:
        return ""
    m = re.search(rf"=== {re.escape(section_name)} ===\n(.*?)(?:\n=== |\Z)", evidence_text, re.DOTALL)
    return m.group(1).strip() if m else ""

format_b["pod_describe"] = format_b["evidence_text"].apply(lambda x: extract_section(x, "kubectl describe pod"))
format_b["pod_logs"] = format_b["evidence_text"].apply(lambda x: extract_section(x, "container logs"))
format_b["pod_logs_previous"] = format_b["evidence_text"].apply(lambda x: extract_section(x, "container logs (previous)"))

# Parse the kubectl describe section for structured fields
parsed_b = format_b["pod_describe"].apply(parse_describe).apply(pd.Series)

for col in parsed_b.columns:
    if col not in format_b.columns:
        format_b[col] = parsed_b[col]
    else:
        format_b[col] = format_b[col].fillna(parsed_b[col])

print(f"Format B (chels + devesh + najel): {len(format_b)} rows")
format_b[["scenario_id", "pod_name", "namespace", "pod_status", "event_reason"]].head()

In [ ]:
# --- Combine into final DataFrame with target columns ---
FINAL_COLS = [
    "scenario_id",
    "namespace",
    "pod_name",
    "service_account_name",
    "node",
    "pod_status",
    "image",
    "container_state",
    "last_state",
    "ready",
    "restart_count",
    "node_selectors",
    "claim_name",
    "event_reason",
    "event_message",
    "pod_describe",
    "pod_logs",
    "pod_logs_previous",
    "evidence_text",
]

combined_df = pd.concat([
    akash_df[FINAL_COLS],
    format_a[FINAL_COLS],
    format_b[FINAL_COLS],
], ignore_index=True)
print(f"Combined: {len(combined_df)} rows")
combined_df.info()

In [ ]:
combined_df.head(10)

In [ ]:
output_path = "data/02-raw/k8s_combined_incidents.jsonl"
combined_df.to_json(output_path, orient="records", lines=True)
print(f"Saved {len(combined_df)} rows to {output_path}")